In [2]:
import torch
from torch import nn
from d2l import torch as d2l

In [3]:
def dropout_layer(x,dropout):
    assert 0<=dropout<=1
    if dropout==1:
        return torch.zeros_like(x)
    if dropout==0:
        return x
    mask=(torch.randn(x.shape)>dropout).float()
    return mask*x/(1-dropout)

In [4]:
num_inputs, num_outputs, num_hiddens1, num_hiddens2 = 784, 10, 256, 256
dropout1, dropout2 = 0.2, 0.5
class Net(nn.Module):
    def __init__(self, num_inputs, num_outputs, num_hiddens1, num_hiddens2, is_training=True):
        super(Net, self).__init__()
        self.num_inputs = num_inputs
        self.training = is_training  # 赋值而不是减法
        
        # 定义全连接层
        self.lin1 = nn.Linear(num_inputs, num_hiddens1)
        self.lin2 = nn.Linear(num_hiddens1, num_hiddens2)
        self.lin3 = nn.Linear(num_hiddens2, num_outputs)
        self.relu = nn.ReLU()
    
    def forward(self, X):
        # 修正：去掉reshape中的空格，并在外部reshape
        X = X.reshape((-1, self.num_inputs))
        
        H1 = self.relu(self.lin1(X))
        if self.training == True:  # 或直接 if self.training
            H1 = dropout_layer(H1, dropout1)
        
        H2 = self.relu(self.lin2(H1))
        if self.training == True:
            H2 = dropout_layer(H2, dropout2)
        
        out = self.lin3(H2)
        return out

# 创建模型实例
net = Net(num_inputs, num_outputs, num_hiddens1, num_hiddens2, is_training=True)
print(net)


Net(
  (lin1): Linear(in_features=784, out_features=256, bias=True)
  (lin2): Linear(in_features=256, out_features=256, bias=True)
  (lin3): Linear(in_features=256, out_features=10, bias=True)
  (relu): ReLU()
)


In [5]:
def train_ch3(net, train_iter, test_iter, loss, num_epochs, updater):
    """训练函数 - 直接复制到 notebook 里用"""
    for epoch in range(num_epochs):
        net.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for X, y in train_iter:
            y_hat = net(X)
            l = loss(y_hat, y)
            
            updater.zero_grad()
            l.backward()
            updater.step()
            
            total_loss += l.item()
            correct += (y_hat.argmax(1) == y).sum().item()
            total += y.shape[0]
        
        # 测试
        net.eval()
        test_correct = 0
        test_total = 0
        with torch.no_grad():
            for X, y in test_iter:
                y_hat = net(X)
                test_correct += (y_hat.argmax(1) == y).sum().item()
                test_total += y.shape[0]
        
        train_acc = correct / total
        test_acc = test_correct / test_total
        print(f'Epoch {epoch+1}: loss={total_loss/len(train_iter):.4f}, '
              f'train_acc={train_acc:.4f}, test_acc={test_acc:.4f}')

In [6]:
num_epochs,lr,batch_size=10,0.5,256
loss=nn.CrossEntropyLoss()
train_iter,test_iter=d2l.load_data_fashion_mnist(batch_size)
trainer=torch.optim.SGD(net.parameters(),lr=lr)
train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer)

Epoch 1: loss=1.0916, train_acc=0.5897, test_acc=0.7031
Epoch 2: loss=0.6482, train_acc=0.7654, test_acc=0.7720
Epoch 3: loss=0.5698, train_acc=0.7970, test_acc=0.8108
Epoch 4: loss=0.5237, train_acc=0.8155, test_acc=0.8153
Epoch 5: loss=0.4992, train_acc=0.8229, test_acc=0.8114
Epoch 6: loss=0.4774, train_acc=0.8291, test_acc=0.8269
Epoch 7: loss=0.4630, train_acc=0.8363, test_acc=0.7901
Epoch 8: loss=0.4485, train_acc=0.8402, test_acc=0.8104
Epoch 9: loss=0.4369, train_acc=0.8436, test_acc=0.8474
Epoch 10: loss=0.4309, train_acc=0.8468, test_acc=0.8446


In [8]:
net = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Dropout(dropout1),
    nn.Linear(256, 256),
    nn.ReLU(),
    nn.Dropout(dropout2),
    nn.Linear(256, 10)
)

def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std=0.01)  # 使用小的标准差，如0.01
        nn.init.constant_(m.bias, 0)  # 可选：初始化偏置

net.apply(init_weights)
trainer=torch.optim.SGD(net.parameters(),lr=lr)
train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer)

Epoch 1: loss=1.1551, train_acc=0.5510, test_acc=0.7610
Epoch 2: loss=0.5731, train_acc=0.7866, test_acc=0.8221
Epoch 3: loss=0.4912, train_acc=0.8212, test_acc=0.7987
Epoch 4: loss=0.4391, train_acc=0.8386, test_acc=0.8369
Epoch 5: loss=0.4182, train_acc=0.8478, test_acc=0.8456
Epoch 6: loss=0.3974, train_acc=0.8540, test_acc=0.8449
Epoch 7: loss=0.3771, train_acc=0.8624, test_acc=0.8486
Epoch 8: loss=0.3626, train_acc=0.8665, test_acc=0.8526
Epoch 9: loss=0.3567, train_acc=0.8706, test_acc=0.8423
Epoch 10: loss=0.3437, train_acc=0.8737, test_acc=0.8577
